# MistralFit — controllo GPU gratuito su Kaggle

Questo notebook controlla una **macchina remota Kaggle**. Il PC personale serve solo come browser.

1. Accedi a [Kaggle](https://www.kaggle.com/code/new) e importa questo notebook.
2. In **Settings → Accelerator** scegli **GPU P100**, se disponibile sul tuo account. Controlla la quota gratuita residua. T4 x2 non è ancora supportata dal nostro protocollo a singola GPU.
3. Esegui la cella e scarica il piccolo file `mistralfit-free-gpu-preflight.json` dal collegamento mostrato.
4. Ferma la sessione GPU per conservare la quota.

**Questa cella non scarica i pesi, non compila CUDA e non esegue inferenza.** Non acquista servizi o crediti. Un esito positivo è solo un controllo preliminare; il confronto Small 4 reale deve ancora essere eseguito.

Le risorse gratuite dipendono dalla disponibilità e dall'account. Può essere richiesta la verifica dell'account. Non attivare piani a pagamento per questa procedura.


In [ ]:
from pathlib import Path
import csv, io, json, os, shutil, subprocess
from datetime import datetime, timezone

# Inventory only: no model download, CUDA compilation, inference, API key or paid service.
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook inside Kaggle, not on your personal computer.')

from IPython.display import FileLink, display

GIB = 1024 ** 3
MODEL_BYTES = 26882944256
MODEL_SHA256 = 'e6363fa561991421a01c665e18b573f3affdef8c1eb039e63da8fcfcb8f826d5'
blockers = []
notes = ['GPU availability and remaining free quota must be checked in Kaggle Settings.',
         'Memory/disk headroom checks are estimates, not proof that the full study will run.',
         'Only this small JSON file is written to the notebook output.']

def run(argv):
    result = subprocess.run(argv, capture_output=True, text=True, timeout=15)
    return {'return_code': result.returncode, 'stdout': result.stdout.strip(), 'stderr': result.stderr.strip()}

mem = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':', 1)
    mem[key] = int(value.split()[0]) * 1024
ram_total, ram_available = mem['MemTotal'], mem.get('MemAvailable', 0)
cap_file = Path('/sys/fs/cgroup/memory.max')
if cap_file.is_file() and cap_file.read_text().strip().isdigit():
    cap = int(cap_file.read_text())
    if 0 < cap < ram_total:
        ram_total = cap
        usage = int(Path('/sys/fs/cgroup/memory.current').read_text())
        # Conservatively include clean inactive file cache when the counter is readable.
        stats = {}
        try:
            stats = dict((k, int(v)) for k, v in (row.split() for row in Path('/sys/fs/cgroup/memory.stat').read_text().splitlines()))
        except (OSError, ValueError):
            pass
        clean = max(0, stats.get('inactive_file', 0) - stats.get('file_dirty', 0) - stats.get('file_writeback', 0))
        ram_available = min(ram_available, max(0, min(cap, cap - usage + min(usage, clean))))

smi = shutil.which('nvidia-smi')
gpus = []
if smi:
    gpu_probe = run([smi, '--query-gpu=index,name,memory.total,memory.free,compute_cap', '--format=csv,noheader,nounits'])
    if gpu_probe['return_code'] == 0:
        for row in csv.reader(io.StringIO(gpu_probe['stdout'])):
            if len(row) != 5:
                continue
            gpus.append({'index': int(row[0]), 'name': row[1].strip(),
                         'total_bytes': int(float(row[2]) * 1024 ** 2),
                         'free_bytes': int(float(row[3]) * 1024 ** 2),
                         'compute_capability': row[4].strip()})
    else:
        blockers.append('nvidia-smi could not provide the required device inventory.')
else:
    blockers.append('No GPU detected. Select a free GPU accelerator in Kaggle Settings.')
if len(gpus) != 1:
    blockers.append('This protocol currently requires one GPU. Choose GPU P100 if available, not T4 x2.')

nvcc = shutil.which('nvcc')
if not nvcc and Path('/usr/local/cuda/bin/nvcc').is_file():
    nvcc = '/usr/local/cuda/bin/nvcc'
compiler = None
if nvcc:
    compiler = {'version': run([nvcc, '--version']), 'architectures': run([nvcc, '--list-gpu-arch'])}
    if compiler['architectures']['return_code'] != 0:
        blockers.append('The CUDA compiler could not report its supported architectures.')
    elif len(gpus) == 1:
        architecture = 'compute_' + gpus[0]['compute_capability'].replace('.', '')
        if architecture not in compiler['architectures']['stdout'].split():
            blockers.append('Installed CUDA compiler does not support this GPU. Do not download weights yet.')
else:
    blockers.append('CUDA nvcc is unavailable; the pinned source-build path needs a compatible toolkit.')

scratch = Path('/kaggle/temp/mistralfit')
scratch.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(scratch)
vram_free = gpus[0]['free_bytes'] if len(gpus) == 1 else 0
if ram_available + vram_free < MODEL_BYTES + 4 * GIB:
    blockers.append('Combined free RAM/VRAM lacks the estimated weight/runtime headroom.')
if disk.free < MODEL_BYTES + 12 * GIB:
    blockers.append('Insufficient free scratch disk for weights, source build and dependencies.')

report = {'scope': 'Remote hardware inventory only. No inference or benchmark was run.',
          'status': 'blocked' if blockers else 'hardware_preflight_passed',
          'created_at': datetime.now(timezone.utc).isoformat(),
          'ram_total_bytes': ram_total, 'ram_available_bytes': ram_available,
          'gpus': gpus, 'cuda_compiler': compiler, 'scratch_free_bytes': disk.free,
          'target_model': {'file': 'mistralai_Mistral-Small-4-119B-2603-IQ1_S.gguf',
                           'bytes': MODEL_BYTES, 'sha256': MODEL_SHA256},
          'blockers': blockers, 'notes': notes}
output = Path('/kaggle/working/mistralfit-free-gpu-preflight.json')
output.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
display(FileLink(str(output)))
print('Stop this session after downloading the small JSON. This cell does not keep a session alive.')


## Risultato e passo successivo

Se trovi `blocked`, conserva il JSON: riporta il motivo e non sono stati scaricati 27 GB di pesi inutilmente. Una P100 richiede un compilatore CUDA compatibile; CUDA 13 ha rimosso la compilazione per le architetture precedenti a 7.5.

Se trovi `hardware_preflight_passed`, il JSON permette di scegliere la compilazione e il carico adatti alla sessione ottenuta. Il protocollo completo e il runner sono nella [guida MistralFit](https://github.com/Berserk-hub150/MistralFit/blob/main/docs/controlled-study.md). Le quote e la sessione vanno verificate prima di avviare il lavoro lungo. Il benchmark non è stato eseguito da questo notebook.

Fonti ufficiali verificate il 15 ottobre 2026:
- [Kaggle — GPU usage](https://www.kaggle.com/docs/efficient-gpu-usage)
- [NVIDIA — GPU notebook su Kaggle](https://docs.nvidia.com/datascience/deployment/26.10/platforms/kaggle/)
- [NVIDIA — supporto delle architetture CUDA](https://developer.nvidia.com/blog/navigating-gpu-architecture-support-a-guide-for-nvidia-cuda-developers/)
